# 3. Unsupervised segmentation — no labels, no GPU

`segment` decomposes a scene into ground and object segments with no training data at all: it filters the ground, describes the rest with normals and features, then grows smooth regions. This is the frugal, laptop-friendly path.

```bash
pip install "geoai3d[laz,viz]"
```

In [ ]:
from geoai3d import read_lidar, segment

cloud = read_lidar("../data/ahn_sample.laz")
labelled = segment(
    cloud,
    ground_resolution=1.0,
    normal_k=16,
    smoothness_degrees=15.0,
    min_region_size=100,
)
segments = labelled.attribute("segment")

Ground is labelled `0`, each grown object region gets its own id from `1` up, and points left unassigned are `-1`.

In [ ]:
import numpy as np

object_ids = np.unique(segments[segments > 0])
print(f"ground points : {100 * (segments == 0).mean():.1f}%")
print(f"object segments: {len(object_ids)}")
print(f"unassigned    : {100 * (segments == -1).mean():.1f}%")

## Visualise the decomposition

Colour by the segment id to see the scene broken into ground and individual objects.

In [ ]:
from geoai3d import view

view(labelled, color_by="segment")

## Under the hood

`segment` composes tools you can also call yourself for finer control — for example, growing regions only on the non-ground points, or clustering them with DBSCAN.

In [ ]:
from geoai3d import ground, estimate_normals, geometric_features
from geoai3d import region_growing

classified = ground(cloud, cloth_resolution=1.0)
objects = classified[~classified.attribute("is_ground")]
objects = estimate_normals(objects, k=16)
objects = geometric_features(objects, k=16)
objects = region_growing(objects, k=16, smoothness_degrees=15.0, min_size=100)
grown = objects.attribute("segment")
print("regions among non-ground points:", len(np.unique(grown[grown >= 0])))

No labels, no model, no GPU — a survey office can run this on a laptop today. Notebook 4 adds supervised labels and exports a GIS layer.